# Daily Outbound Quantity Forecast

FastAPI + MySQL WMS project data pipeline for product-level daily outbound quantity forecasting.

## 0. Install dependencies if needed

In [ ]:
# Uncomment and run this cell if these packages are not installed in your notebook kernel.
# %pip install pandas numpy scikit-learn SQLAlchemy PyMySQL python-dotenv matplotlib

## 1. Imports and configuration

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from dotenv import dotenv_values
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Project path. Change this if you move the project.
PROJECT_ROOT = Path(r"C:\Users\hi\Desktop\개인 프로젝트\WMS_project")
BACKEND_ROOT = PROJECT_ROOT / "backend"

# The FastAPI project uses DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME.
# This notebook reads backend/.env first, then project/.env, then the current environment.
env = {}
for env_path in [BACKEND_ROOT / ".env", PROJECT_ROOT / ".env", Path.cwd() / ".env"]:
    if env_path.exists():
        env.update(dotenv_values(env_path))

def get_config(name, default=None):
    return os.getenv(name) or env.get(name) or default

db_user = get_config("DB_USER")
db_password = get_config("DB_PASSWORD")
db_host = get_config("DB_HOST", "localhost")
db_port = int(get_config("DB_PORT", 3306))
db_name = get_config("DB_NAME")

missing = [
    name
    for name, value in {
        "DB_USER": db_user,
        "DB_PASSWORD": db_password,
        "DB_HOST": db_host,
        "DB_PORT": db_port,
        "DB_NAME": db_name,
    }.items()
    if value in (None, "")
]
if missing:
    raise ValueError(f"Missing DB configuration values: {missing}")

DB_URL = URL.create(
    drivername="mysql+pymysql",
    username=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    database=db_name,
)

# Add known event / promotion / special operation periods here.
# Each tuple is inclusive: (start_date, end_date).
EVENT_PERIODS = [
    # ("2026-01-01", "2026-01-03"),
    # ("2026-09-24", "2026-09-30"),
]

# Test set uses the latest N calendar dates across the completed panel.
TEST_DAYS = 14
RANDOM_STATE = 42

print(f"DB target: mysql+pymysql://{db_user}:***@{db_host}:{db_port}/{db_name}")

## 2. Load outbounds from MySQL

In [ ]:
engine = create_engine(DB_URL, pool_pre_ping=True)

query = text("""
    SELECT
        DATE(outbound_date) AS outbound_date,
        product_id,
        location_id,
        outbound_qty
    FROM outbounds
    WHERE outbound_date IS NOT NULL
""")

outbounds = pd.read_sql(query, engine)
outbounds["outbound_date"] = pd.to_datetime(outbounds["outbound_date"])

print(f"Loaded rows: {len(outbounds):,}")
display(outbounds.head())

## 3. Aggregate daily outbound quantity by date and product

In [ ]:
if outbounds.empty:
    raise ValueError("The outbounds table returned no rows.")

daily = (
    outbounds
    .groupby(["outbound_date", "product_id"], as_index=False)["outbound_qty"]
    .sum()
    .rename(columns={"outbound_qty": "daily_outbound_qty"})
    .sort_values(["product_id", "outbound_date"])
    .reset_index(drop=True)
)

display(daily.head(10))

## 4. Complete daily date range for each product

In [ ]:
def complete_product_calendar(group: pd.DataFrame) -> pd.DataFrame:
    product_id = group.name
    date_index = pd.date_range(
        start=group["outbound_date"].min(),
        end=group["outbound_date"].max(),
        freq="D",
    )

    completed = (
        group
        .set_index("outbound_date")
        .reindex(date_index)
        .rename_axis("outbound_date")
        .reset_index()
    )
    completed["product_id"] = product_id
    completed["daily_outbound_qty"] = completed["daily_outbound_qty"].fillna(0).astype(int)
    return completed

daily_full = (
    daily
    .groupby("product_id", group_keys=False)
    .apply(complete_product_calendar)
    .sort_values(["product_id", "outbound_date"])
    .reset_index(drop=True)
)

print(f"Aggregated rows before calendar fill: {len(daily):,}")
print(f"Rows after calendar fill: {len(daily_full):,}")
display(daily_full.head(10))

## 5. Create time-series features

In [ ]:
def is_event_period(date_value) -> int:
    date_value = pd.Timestamp(date_value)
    for start, end in EVENT_PERIODS:
        if pd.Timestamp(start) <= date_value <= pd.Timestamp(end):
            return 1
    return 0

features = daily_full.copy()

# Lag features are computed per product and use only previous values.
product_qty = features.groupby("product_id")["daily_outbound_qty"]
features["lag_1"] = product_qty.shift(1)
features["lag_7"] = product_qty.shift(7)
features["rolling_7_mean"] = features.groupby("product_id")["daily_outbound_qty"].transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=1).mean()
)

# Calendar and event features.
features["day_of_week"] = features["outbound_date"].dt.dayofweek
features["month"] = features["outbound_date"].dt.month
features["is_event_period"] = features["outbound_date"].apply(is_event_period).astype(int)

# Cold-start dates have no prior lag values. Use 0 because no known prior outbound exists in this panel.
lag_columns = ["lag_1", "lag_7", "rolling_7_mean"]
features[lag_columns] = features[lag_columns].fillna(0)

display(features.head(15))

## 6. Split train/test by date

In [ ]:
unique_dates = pd.Series(features["outbound_date"].sort_values().unique())
if len(unique_dates) < 2:
    raise ValueError("At least two unique dates are required for a date-based train/test split.")

test_size = min(TEST_DAYS, max(1, len(unique_dates) // 5))
test_start_date = pd.Timestamp(unique_dates.iloc[-test_size])

train_df = features[features["outbound_date"] < test_start_date].copy()
test_df = features[features["outbound_date"] >= test_start_date].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Train/test split produced an empty set. Reduce TEST_DAYS or check the data range.")

feature_cols = [
    "product_id",
    "lag_1",
    "lag_7",
    "rolling_7_mean",
    "day_of_week",
    "month",
    "is_event_period",
]
target_col = "daily_outbound_qty"

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"Date range: {features['outbound_date'].min().date()} to {features['outbound_date'].max().date()}")
print(f"Test start date: {test_start_date.date()}")
print(f"Train rows: {len(train_df):,}, Test rows: {len(test_df):,}")

## 7. Train baseline and RandomForestRegressor

In [ ]:
# Baseline: tomorrow equals yesterday, using lag_1.
baseline_pred = test_df["lag_1"].to_numpy()

rf_model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# Quantity predictions should not be negative. Random Forest usually will not go below zero here,
# but clipping keeps the output valid if the model or target changes later.
rf_pred = np.clip(rf_pred, 0, None)

## 8. Compare MAE and RMSE

In [ ]:
def evaluate_predictions(y_true, y_pred) -> dict:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    }

metrics_df = pd.DataFrame(
    [
        {"model": "Baseline_lag_1", **evaluate_predictions(y_test, baseline_pred)},
        {"model": "RandomForestRegressor", **evaluate_predictions(y_test, rf_pred)},
    ]
).sort_values("MAE")

display(metrics_df)

## 9. Compare predictions with actual values

In [ ]:
comparison_df = (
    test_df[["outbound_date", "product_id", "daily_outbound_qty"]]
    .rename(columns={"daily_outbound_qty": "actual_daily_outbound_qty"})
    .assign(
        baseline_prediction=baseline_pred,
        rf_prediction=rf_pred,
        baseline_error=lambda df: df["actual_daily_outbound_qty"] - df["baseline_prediction"],
        rf_error=lambda df: df["actual_daily_outbound_qty"] - df["rf_prediction"],
    )
    .sort_values(["outbound_date", "product_id"])
    .reset_index(drop=True)
)

display(comparison_df.head(50))

## Optional: Feature importance

In [ ]:
feature_importance_df = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": rf_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance_df)